In [1]:
import os
import simo
from simo import load_data, process_anndata, find_marker, alignment_1, assign_coord_1
from simo.regulation import regulation_analysis, spatial_regulation
from simo import sdplot, sfplot, cor_plot, module_vilolin_plot, module_dot_plot, module_pca_plot, module_spatial_plot, module_heatmap_plot, spatial_lineplot, plot_3d
from simo.helper import extract_reduction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import itertools
from tqdm import tqdm
import random
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import issparse
import scipy
from scanpy import AnnData
import time

import warnings
warnings.filterwarnings("ignore")

In [2]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    #torch.manual_seed(seed)
    #if torch.cuda.is_available():
    #    torch.cuda.manual_seed_all(seed)
    
    
def integer_allocation(prop, counts):
    expected_cells = prop * counts[:, None]
    int_cells = np.floor(expected_cells).astype(int)
    frac_cells = expected_cells - int_cells
    remaining_cells = counts - int_cells.sum(axis=1)
    for i in range(len(counts)):
        frac_order = np.argsort(-frac_cells[i])
        for j in range(remaining_cells[i]):
            int_cells[i, frac_order[j]] += 1
    return int_cells


def jitter_coord(coord):
    # cell number
    num = coord.shape[0]
    # min distance
    coord_unique = np.unique(coord, axis=0)
    nbrs = NearestNeighbors(n_neighbors=2).fit(coord_unique)
    distances, indices = nbrs.kneighbors(coord_unique)
    min_distance = min(distances[:, -1][distances[:, -1] > 0])

    x_list = list(coord[:, 0])
    y_list = list(coord[:, 1])

    set_seed(0)
    length = np.random.uniform(0, min_distance, num)
    radius = np.pi * np.random.uniform(0, 2, num)

    x_list_new = x_list + length * np.cos(radius)
    y_list_new = y_list + length * np.sin(radius)
    coord_new = np.array([[x_list_new[i], y_list_new[i]] for i in range(num)])

    return coord_new


def adjust_abundance(
    adata_st: AnnData,
    adata_sc: AnnData,
    celltype_key: str = 'celltype',
):
    celltype_unique = sorted(set(adata_sc.obs[celltype_key]))
    
    cell_counts = np.array(adata_st.obs['estimated_cell_number'])
    prop = np.array(adata_st.obs[celltype_unique].copy())
    map_target = integer_allocation(prop, cell_counts)
    
    target_num_list = map_target.sum(axis=0)
    sc_num_list = np.array(adata_sc.obs[celltype_key].value_counts()[celltype_unique])
    diff_num_list = sc_num_list - target_num_list
    
    adata_list = []
    for i in range(len(celltype_unique)):
        adata_tmp = adata_sc[adata_sc.obs[celltype_key] == celltype_unique[i]].copy()
        adata_list.append(adata_tmp)
        
    print(f"Adjust abundance of each cell types")
    for i in tqdm(range(len(celltype_unique))):
        adjust_num = diff_num_list[i]
        adata_tmp = adata_list[i].copy()
        if adjust_num >= 0:
            set_seed(0)
            selected_indices = np.random.choice(adata_tmp.shape[0], size=target_num_list[i], replace=False)
            adata_tmp = adata_tmp[selected_indices]
        elif adjust_num < 0:
            fold = np.abs(diff_num_list[i]) / sc_num_list[i]
            if fold > 1:
                fold_int = int(fold)
                selected_indices = list(range(adata_tmp.shape[0])) * fold_int
                set_seed(0)
                selected_indices2 = list(
                    np.random.choice(adata_tmp.shape[0], size=(np.abs(diff_num_list[i]) - fold_int * adata_tmp.shape[0]), replace=False)
                )
                selected_indices.extend(selected_indices2)
                selected_indices = np.array(selected_indices)
            else:
                set_seed(0)
                selected_indices = np.random.choice(adata_tmp.shape[0], size=np.abs(diff_num_list[i]), replace=False)
            
            adata_tmp_replicate = adata_tmp[selected_indices].copy()
            adata_tmp = sc.concat([adata_tmp, adata_tmp_replicate]).copy()
            
        adata_list[i] = adata_tmp.copy()
        
    adata_sc_new = sc.concat(adata_list).copy()
    adata_sc_new.obs_names_make_unique()
    print(f"Done")
    
    return adata_st, adata_sc_new


def process_result(
    adata_st: AnnData,
    adata_sc: AnnData,
    transport_matrix: np.array,
    celltype_key: str = 'celltype',
):
    
    print(f"Assign cells")
    spot_to_cells = []
    cell_counts = adata_st.obs['estimated_cell_number']
    for i in tqdm(range(transport_matrix.shape[1])):
        k = int(cell_counts[i])
        top_k_cells = np.argsort(-transport_matrix[:, i])[:k]
        spot_to_cells.append(list(top_k_cells))
        
    print(f"Create new data")
    if issparse(adata_sc.X):
        adata_sc.X = adata_sc.X.toarray()
        
    # original
    original_spot = list(adata_st.obs_names)
    original_cell = list(adata_sc.obs_names)
    original_celltype = list(adata_sc.obs[celltype_key])
    original_x = list(adata_st.obsm['spatial'][:, 0])
    original_y = list(adata_st.obsm['spatial'][:, 1])
    original_expr = adata_sc.X
    
    # new
    cell_list = []
    celltype_list = []
    spot_list = []
    x_list = []
    y_list = []
    expr_list = []
    
    for i, indices in enumerate(spot_to_cells):
        cell_list.extend(original_cell[idx] for idx in indices)
        celltype_list.extend(original_celltype[idx] for idx in indices)

        spot_list.extend([original_spot[i]] * len(indices))
        x_list.extend([original_x[i]] * len(indices))
        y_list.extend([original_y[i]] * len(indices))

        expr_list.extend(original_expr[indices])
        
    new_id_list = ['CID' + str(i + 1) for i in range(len(cell_list))]
    
    new_meta = pd.DataFrame({
        'NewCID': new_id_list,
        'OriginalCID': cell_list,
        'CellType': celltype_list,
        'SpotID': spot_list,
        'X': x_list,
        'Y': y_list,
    })
    
    new_meta.index = new_id_list
    new_expr = np.array(expr_list)
    new_expr = scipy.sparse.csr_matrix(new_expr)
    coord = np.array(new_meta[['X', 'Y']])
    coord_jitter = jitter_coord(coord)
    
    new_meta['X_jitter'] = coord_jitter[:, 0]
    new_meta['Y_jitter'] = coord_jitter[:, 1]

    # new AnnData
    adata_new = sc.AnnData(new_expr)
    adata_new.obs = new_meta
    adata_new.obs_names = new_id_list
    adata_new.var_names = adata_sc.var_names
    adata_new.obsm['spatial'] = coord_jitter
    
    print("Done")
    return adata_new


In [ ]:
noise_list = ['0', '05', '10', '20', '40']
n_list = [5, 10, 15]

for noise in noise_list:
    for n in n_list:
        ad_sc = sc.read('../output/Cerebellum_sc_noise' + noise + '.h5ad')
        ad_sp = sc.read('../output/Cerebellum_st_n' + str(n) + '.h5ad')
        
        # spot_x, spot_y, spot
        ad_sp.obs = ad_sp.obs.iloc[:, -3:]
        
        # load deconvolution results
        prop = pd.read_csv('../results/C2L_Noise' + noise + '_n' + str(n) + '.csv', index_col=0)
        ct_sort = sorted(set(ad_sc.obs['CellType']))
        prop = prop[ct_sort]
        prop = prop.loc[ad_sp.obs_names]
        obs_raw = ad_sp.obs.copy()
        obs_new = pd.concat([obs_raw, prop], axis=1)
        ad_sp.obs = obs_new
        
        # scPositioner results
        scpositioner_res = sc.read('../results/Cerebellum_scPositioner_noise' + noise + '_n' + str(n) + '.h5ad')
        ad_sp.obs["estimated_cell_number"] = np.array(scpositioner_res.obs['SpotID'].value_counts()[ad_sp.obs_names])
        
        ad_sp, ad_sc = adjust_abundance(
            adata_st = ad_sp.copy(),
            adata_sc = ad_sc.copy(),
            celltype_key='CellType',
        )
        
        sc.tl.pca(ad_sc, n_comps=30)
        ad_sc.obsm['reduction'] = ad_sc.obsm['X_pca'].copy()
        ad_sc.obs['type'] = ad_sc.obs['CellType'].copy()
        ad_sp.obs['type'] = 'uniform_region'
        expected_num = ad_sp.obs['estimated_cell_number']
        expected_num = pd.DataFrame(expected_num)
        expected_num.columns = ['cell_num']
        
        ad_sc = process_anndata(ad_sc)
        ad_sp = process_anndata(ad_sp)
        
        alignment_result = alignment_1(adata1=ad_sp, adata2=ad_sc, alpha=0.1, aware_st=False, aware_sc=True)
        spatial_assignment = assign_coord_1(adata1=ad_sp, adata2=ad_sc, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)
        
        adata_new = ad_sc[spatial_assignment['cell'], :]
        spatial_assignment = spatial_assignment.set_index('cell')
        adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
        adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
        adata_new.var_names = [item.upper() for item in adata_new.var_names]
        adata_new.write('../results/Cerebellum_SIMO_noise' + noise + '_n' + str(n) + '.h5ad')

Adjust abundance of each cell types


100%|██████████| 11/11 [00:01<00:00, 10.89it/s]


Done
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Calculating dissimilarity using euclidean distance on scaled data...
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Processing completed.
Constructing connectivity...
k = 10
aware_sc = True
aware power = 2
Running OT...
alpha = 0.1
OT done!
Assigning spatial coordinates to cells...
random =

100%|██████████| 11/11 [00:03<00:00,  3.20it/s]


Done
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Calculating dissimilarity using euclidean distance on scaled data...
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Processing completed.
Constructing connectivity...
k = 10
aware_sc = True
aware power = 2
Running OT...
alpha = 0.1
OT done!
Assigning spatial coordinates to cells...
random =

...

Done
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Calculating dissimilarity using euclidean distance on scaled data...
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Processing completed.
Constructing connectivity...
k = 10
aware_sc = True
aware power = 2
Running OT...
alpha = 0.1
OT done!
Assigning spatial coordinates to cells...
random =

100%|██████████| 11/11 [00:02<00:00,  3.72it/s]


Done
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Calculating dissimilarity using euclidean distance on scaled data...
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Processing completed.
Constructing connectivity...
k = 10
aware_sc = True
aware power = 2
Running OT...
alpha = 0.1
OT done!
Assigning spatial coordinates to cells...
random =

100%|██████████| 11/11 [00:02<00:00,  4.49it/s]


Done
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Performing PCA...
Calculating neighbors based on cosine metric...
Performing UMAP...
Processing completed.
Calculating dissimilarity using euclidean distance on scaled data...
Processing RNA data...
Identifying highly variable genes...
Normalizing total counts...
Applying log1p transformation...
Saving pre-log1p counts to a layer...
Scaling the data...
Processing completed.
Constructing connectivity...
k = 10
aware_sc = True
aware power = 2
Running OT...
alpha = 0.1
